
# 01 — Dataset Exploration

## Objective

The objective of this notebook is to explore and understand the
intracranial EEG (iEEG) dataset before performing signal preprocessing
or statistical analysis.

The exploration includes:

- Understanding the dataset structure.
- Inspecting participants and recordings.
- Exploring recording metadata.
- Understanding channel information.
- Exploring stimulation events.
- Selecting a representative recording.
- Loading and visualising the raw signal.

## Dataset

- Dataset ID: ds006519
- Modality: Intracranial EEG (iEEG)
- Format: BIDS
- Source: https://openneuro.org/datasets/ds006519

## Important note

This notebook is exploratory. No signal filtering, artefact removal,
or scientific conclusions will be performed at this stage.



## 1. Import libraries

We use Python libraries to inspect the dataset, manipulate metadata,
and visualise the recordings.

- pathlib: navigate directories and locate files.
- json: read JSON metadata files.
- pandas: work with tabular data.
- numpy: numerical operations.
- matplotlib: visualise data.
- mne: load and inspect electrophysiological recordings.


In [2]:

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mne


In [3]:

# Display library versions

print(f"MNE version: {mne.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")


MNE version: 1.13.2
NumPy version: 2.5.5
Pandas version: 3.1.1


In [6]:

# Define the project and dataset paths

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"

# Check that the dataset directory exists

assert RAW_DATA_DIR.exists(), "Raw dataset directory was not found."

print(f"Project directory: {PROJECT_ROOT}")
print(f"Raw dataset directory: {RAW_DATA_DIR}")


Project directory: /workspaces/seeg-signal-analysis
Raw dataset directory: /workspaces/seeg-signal-analysis/data/raw



## 2. Dataset overview

Before selecting an individual recording, we first investigate
the overall dataset composition.

We will identify:

- The number of participants.
- The number of iEEG recordings.
- The available recording formats.
- The distribution of recordings across participants.

This helps us understand the dataset and select a representative
recording for further exploration.


In [7]:

participants_file = RAW_DATA_DIR / "participants.tsv"

participants = pd.read_csv(participants_file, sep="\t")

print(f"Number of participants: {len(participants)}")

display(participants.head())


Number of participants: 23


,participant_id,seeg_id,age,sex,excluded_from_network_analysis,reason_for_exclusion
0,sub-01,12,35,f,0,NaN
1,sub-02,13,24,f,0,NaN
2,sub-03,17,33,m,0,NaN
3,sub-04,18,11,f,0,NaN
4,sub-05,19,9,f,1,"Modified anatomy, freesurfer analysis not poss..."


In [ ]:

# Find all iEEG directories

ieeg_directories = sorted(RAW_DATA_DIR.glob("sub-*/ses-*/ieeg"))


print(f"Participants with iEEG directories: {len(ieeg_directories)}")

for directory in ieeg_directories:
    print(directory)


Participants with iEEG directories: 35
/workspaces/seeg-signal-analysis/data/raw/sub-01/ses-01/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-02/ses-01/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-03/ses-01/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-03/ses-02/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-03/ses-03/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-03/ses-04/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-04/ses-01/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-04/ses-02/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-06/ses-01/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-06/ses-02/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-07/ses-01/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-07/ses-02/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-07/ses-03/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-08/ses-01/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-08/ses-02/ieeg
/workspaces/seeg-signal-analysis/data/raw/sub-

In [63]:

# Identify common electrophysiological recording formats

signal_extensions = [".edf", ".vhdr", ".set", ".fif"]


recording_files = sorted(
    file
    for directory in ieeg_directories
    for file in directory.iterdir()
    if file.suffix.lower() in signal_extensions
)

print(f"Number of recording files: {len(recording_files)}")

for file in recording_files[:20]:
    print(file)


Number of recording files: 35
/workspaces/seeg-signal-analysis/data/raw/sub-01/ses-01/ieeg/sub-01_ses-01_task-dcs_ieeg.vhdr
/workspaces/seeg-signal-analysis/data/raw/sub-02/ses-01/ieeg/sub-02_ses-01_task-dcs_ieeg.vhdr
/workspaces/seeg-signal-analysis/data/raw/sub-03/ses-01/ieeg/sub-03_ses-01_task-dcs_ieeg.vhdr
/workspaces/seeg-signal-analysis/data/raw/sub-03/ses-02/ieeg/sub-03_ses-02_task-dcs_ieeg.vhdr
/workspaces/seeg-signal-analysis/data/raw/sub-03/ses-03/ieeg/sub-03_ses-03_task-dcs_ieeg.vhdr
/workspaces/seeg-signal-analysis/data/raw/sub-03/ses-04/ieeg/sub-03_ses-04_task-dcs_ieeg.vhdr
/workspaces/seeg-signal-analysis/data/raw/sub-04/ses-01/ieeg/sub-04_ses-01_task-dcs_ieeg.vhdr
/workspaces/seeg-signal-analysis/data/raw/sub-04/ses-02/ieeg/sub-04_ses-02_task-dcs_ieeg.vhdr
/workspaces/seeg-signal-analysis/data/raw/sub-06/ses-01/ieeg/sub-06_ses-01_task-dcs_ieeg.vhdr
/workspaces/seeg-signal-analysis/data/raw/sub-06/ses-02/ieeg/sub-06_ses-02_task-dcs_ieeg.vhdr
/workspaces/seeg-signal-analys

## 2. Recording metadata

The recording metadata describes the technical properties of the
iEEG acquisition, including sampling frequency, measurement units,
and recording reference.

Understanding these properties is essential before preprocessing
or analysing the signal.

In [64]:
ieeg_metadata_path = (
    RAW_DATA_DIR
    / "sub-01"
    / "ses-01"
    / "ieeg"
    / "sub-01_ses-01_task-dcs_ieeg.json"
)

with open(ieeg_metadata_path, "r") as f:
    ieeg_metadata = json.load(f)

ieeg_metadata

{'TaskName': 'dcs',
 'SamplingFrequency': 4096,
 'PowerLineFrequency': 50,
 'SoftwareFilters': 'n/a',
 'HardwareFilters': 'n/a',
 'Manufacturer': 'Nicolet',
 'ManufacturersModelName': 'Wireless64',
 'TaskDescription': 'Direct cortical stimulation evoking negative motor responses',
 'Instructions': '',
 'InstitutionName': 'University Emergency Hopital Bucharest',
 'InstitutionAddress': 'Bucharest, Romania',
 'ECOGChannelCount': 0,
 'SEEGChannelCount': 31,
 'EEGChannelCount': 0,
 'EOGChannelCount': 0,
 'ECGChannelCount': 0,
 'EMGChannelCount': 0,
 'MiscChannelCount': 0,
 'TriggerChannelCount': 0,
 'RecordingDuration': 32,
 'RecordingType': 'continuous',
 'SubjectArtefactDescription': 'High-frequency cortical stimulation artefact',
 'SoftwareVersions': '',
 'iEEGReference': 'intracranial white matter electrode',
 'ElectrodeManufacturer': 'DIXI Medical',
 'ElectrodeManufacturersModelName': 'MicroDeep',
 'iEEGGround': 'intracranial white matter electrode',
 'iEEGPlacementScheme': '',
 'iEEG

In [65]:
print("Sampling frequency:", ieeg_metadata.get("SamplingFrequency"))
print("Power line frequency:", ieeg_metadata.get("PowerLineFrequency"))
print("Recording reference:", ieeg_metadata.get("iEEGReference"))
print("Recording units:", ieeg_metadata.get("iEEGGround"))

Sampling frequency: 4096
Power line frequency: 50
Recording reference: intracranial white matter electrode
Recording units: intracranial white matter electrode


## 3. Channel metadata

The channels.tsv file describes the individual
recorded channels.

We inspect channel names, types, units and status
to understand which signals are available and
whether any channels have been marked as problematic.

In [66]:
channels_path = (
    RAW_DATA_DIR
    / "sub-01"
    / "ses-01"
    / "ieeg"
    / "sub-01_ses-01_task-dcs_channels.tsv"
)

channels = pd.read_csv(channels_path, sep="\t")

print("Number of channels:", len(channels))
print("\nAvailable columns:")
print(channels.columns.tolist())

display(channels)

Number of channels: 31

Available columns:
['name', 'type', 'units', 'low_cutoff', 'high_cutoff', 'reference', 'group', 'sampling_frequency', 'description', 'notch', 'status', 'status_description']


,name,type,units,low_cutoff,high_cutoff,reference,group,sampling_frequency,description,notch,status,status_description
0,B'02-B'03,SEEG,µV,NaN,NaN,intracranial,B',4096,EEG,NaN,good,NaN
1,B'03-B'04,SEEG,µV,NaN,NaN,intracranial,B',4096,EEG,NaN,good,NaN
2,B'11-B'12,SEEG,µV,NaN,NaN,intracranial,B',4096,EEG,NaN,good,NaN
3,A'01-A'02,SEEG,µV,NaN,NaN,intracranial,A',4096,EEG,NaN,good,NaN
4,A'02-A'03,SEEG,µV,NaN,NaN,intracranial,A',4096,EEG,NaN,good,NaN
5,A'03-A'04,SEEG,µV,NaN,NaN,intracranial,A',4096,EEG,NaN,good,NaN
6,A'13-A'14,SEEG,µV,NaN,NaN,intracranial,A',4096,EEG,NaN,good,NaN
7,I'01-I'02,SEEG,µV,NaN,NaN,intracranial,I',4096,EEG,NaN,good,NaN
8,I'04-I'05,SEEG,µV,NaN,NaN,intracranial,I',4096,EEG,NaN,good,NaN
9,I'07-I'08,SEEG,µV,NaN,NaN,intracranial,I',4096,EEG,NaN,good,NaN
